# Unshifted residual-width ratio for AR order selection

The earlier ratio based on $\widehat W_d=\widehat m_d/|\widehat u_{d,0}|$ is unstable when a minimizing direction is nearly shifted ($u_0\approx0$). Here we **replace** that spherical optimization with the unshifted residual-span problem

$$
\widehat R_d=\min_{\beta\in\mathbb R^{d-1}}\left[\max_t\left(X_t-\sum_{j=1}^{d-1}\beta_jX_{t-j}\right)-\min_t\left(X_t-\sum_{j=1}^{d-1}\beta_jX_{t-j}\right)\right].
$$

This is the same point-cloud width functional with the current-time coefficient fixed at one: $\widehat R_d=\min_{u_0\ne0}\widehat w_d(u)/|u_0|$. It directly optimizes the ratio instead of dividing the geometric minimum by one arbitrarily chosen minimizer coordinate. A free residual center makes the objective a linear program. It is related to bounded-innovation fitting in [Horváth (1989)](https://dml.cz/handle/10338.dmlcz/104367), but this free-center span objective and selector are our exploratory adaptation, not a theorem from that paper.

Use the same terminal-excess statistic as before, now with $R_d$ in place of $W_d$:

$$
T_d^{(R)}=\frac{\widehat R_d-\widehat R_D}{(D-d-1)^{-1}\sum_{j=d+1}^{D-1}(\widehat R_j-\widehat R_D)},\qquad \widehat p=\operatorname*{arg\,max}_{1\le d\le D-2}T_d^{(R)},\qquad D=10.
$$

Every order uses the **same time indices** $t=D,\ldots,T$, so the feasible coefficient sets are nested and the fitted $\widehat R_d$ are nonincreasing, up to numerical solver error. Consequently the denominator is nonnegative without taking an absolute value. This follows directly from the displayed optimization problem. Ratio-based cutoff estimation has a precedent in other settings, such as the [eigenvalue-ratio test of Ahn and Horenstein (2013)](https://doi.org/10.3982/ECTA8968), but their consistency result does not apply here.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from threadpoolctl import threadpool_limits
from IPython.display import display

AR_DIR = next(
    candidate.resolve()
    for root in (Path.cwd(), *Path.cwd().parents)
    for candidate in (root / "Code" / "Code_AR_processes", root)
    if (candidate / "ar_tools/functions.py").is_file()
)
PROJECT_ROOT = AR_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.ar_tools.functions import (
    jones_ar_coefficients, simulate_uniform_ar,
)

D = 10
SAMPLE_SIZE = 2_000
SEED = 20260816
REPLICATIONS = 50
KNOWN_INNOVATION_WIDTH = 2.0
CANDIDATE_ORDERS = np.arange(1, D - 1)  # 1,...,8


In [2]:
from Code.Code_AR_processes.ar_tools.lp_widths import unshifted_residual_widths


from Code.Code_AR_processes.ar_tools.breakpoint_selectors import select_tail_ratio


from Code.Code_AR_processes.ar_tools.classical_selectors import select_bic


In [3]:
rows = []
with threadpool_limits(limits=1):
    for true_order in range(1, 7):
        for process_id in range(REPLICATIONS):
            process_rng = np.random.default_rng(
                np.random.SeedSequence([SEED, process_id, 0])
            )
            coefficients = jones_ar_coefficients(true_order, rng=process_rng)
            series = simulate_uniform_ar(
                coefficients,
                sample_size=SAMPLE_SIZE,
                burnin=1_000,
                half_width=KNOWN_INNOVATION_WIDTH / 2,
                rng=process_rng,
            )
            widths, y, lag_matrix = unshifted_residual_widths(series, max_dimension=D)
            selected_order, scores = select_tail_ratio(widths, max_dimension=D, candidate_orders=CANDIDATE_ORDERS)
            known_width_order = int(CANDIDATE_ORDERS[
                np.argmin(np.abs(widths[CANDIDATE_ORDERS] - KNOWN_INNOVATION_WIDTH))
            ])
            rows.append({
                "true_order": true_order,
                "process": process_id,
                "ratio_order": selected_order,
                "known_width_order": known_width_order,
                "bic_order": select_bic(y, lag_matrix, candidate_orders=CANDIDATE_ORDERS),
                "nonmonotone": bool(np.any(np.diff(widths) > 1e-8)),
                "R_at_breakpoint": widths[true_order],
                "R_terminal": widths[-1],
                "plateau_range": np.ptp(widths[true_order:]),
            })

results = pd.DataFrame(rows)
assert len(results) == 6 * REPLICATIONS

summary_rows = []
for true_order, group in results.groupby("true_order"):
    for label, column in [
        (r"Tail ratio $T_d^{(R)}$", "ratio_order"),
        (r"Known $W=2$ (oracle)", "known_width_order"),
        ("BIC (benchmark)", "bic_order"),
    ]:
        selected = group[column]
        summary_rows.append({
            "True order": true_order,
            "Selector": label,
            "Correct": int((selected == true_order).sum()),
            "Under": int((selected < true_order).sum()),
            "Over": int((selected > true_order).sum()),
            "Undefined": int(selected.isna().sum()),
        })
summary = pd.DataFrame(summary_rows).set_index(["True order", "Selector"])
display(summary.style.set_caption(
    "50 Jones-sampled AR(p) processes per order; T=2000, D=10, innovations Uniform[-1,1]"
))
print(
    f"Width curves with an upward step larger than 1e-8: "
    f"{int(results['nonmonotone'].sum())}/{len(results)}; "
    f"median fitted plateau range: {results['plateau_range'].median():.5f}."
)


Correct  Under  Over  Undefined
True order Selector                                               
1          Tail ratio $T_d^{(R)}$       45      0     5          0
           Known $W=2$ (oracle)         50      0     0          0
           BIC (benchmark)              50      0     0          0
2          Tail ratio $T_d^{(R)}$       39      9     2          0
           Known $W=2$ (oracle)         50      0     0          0
           BIC (benchmark)              44      6     0          0
3          Tail ratio $T_d^{(R)}$       41      8     1          0
           Known $W=2$ (oracle)         49      1     0          0
           BIC (benchmark)              43      7     0          0
4          Tail ratio $T_d^{(R)}$       44      6     0          0
           Known $W=2$ (oracle)         48      2     0          0
           BIC (benchmark)              45      5     0          0
5          Tail ratio $T_d^{(R)}$       47      2     1          0
           Known $W=2$ (oracle)         50      0     0          0
           BIC (benchmark)              46      4     0          0
6          Tail ratio $T_d^{(R)}$       48      2     0          0
           Known $W=2$ (oracle)         48      2     0          0
           BIC (benchmark)              46      4     0          0

Width curves with an upward step larger than 1e-8: 0/300; median fitted plateau range: 0.00393.


The proposed ratio selects the correct order in **264/300** simulated processes (50 per order $p=1,\ldots,6$), compared with **274/300** for the no-intercept BIC benchmark and **295/300** for the oracle that knows the innovation width $W=2$. On the 50 AR(3) processes using the earlier notebook's seeds, it is correct in **41/50** cases, compared with **1/50** for $T_d$ built from $\widehat m_d/|\widehat u_{d,0}|$. This is an empirical improvement, not a proved guarantee.

The normalization is essential: $\widehat R_d$ is obtained by **replacing** the original spherical minimum-width computation with a linear program. It uses no additional OLS fit for the proposed selector, but it is not merely a byproduct of the old optimizer's minimizing direction. BIC is computed only as a benchmark. The ratio itself uses no penalty, confidence level, or denominator tolerance; $D=10$ is still a chosen maximum dimension.

A consistency proof would need more than $\widehat R_d\to R_d$. In particular, after the true breakpoint one must control ratios of two vanishing empirical excesses, so that a spurious large post-breakpoint $T_d^{(R)}$ cannot beat $T_p^{(R)}$. The positive denominator and well-behaved finite-sample curve remove the two observed failures of the previous construction, but do not by themselves establish that stochastic rate condition. This is our inference from the displayed statistic and experiment, not a claim made by [Horváth (1989)](https://dml.cz/handle/10338.dmlcz/104367) or [Ahn and Horenstein (2013)](https://doi.org/10.3982/ECTA8968). [Yi and Neykov](https://arxiv.org/abs/2108.07630) give non-asymptotic bounds for related $L^\infty$ regression with uniform noise, but their independent-design setting does not automatically cover autoregressive lag vectors or this ratio statistic.
